In [3]:
// Minimal candle test without GUI
:dep candle-core = { path = "../../candle-core", default-features = false }
:dep image = "0.25"
:dep anyhow = "1.0"
use candle_core::{Tensor, Device, DType};
use anyhow::Result;

fn save_image<P: AsRef<std::path::Path>>(img: &Tensor, p: P) -> Result<()> {
    let p = p.as_ref();
    let (c,h,w) = img.dims3()?; if c!=3 { anyhow::bail!("expected (3,H,W)"); }
    let img2 = img.permute((1,2,0))?.flatten_all()?;
    let pixels = img2.to_vec1::<u8>()?;
    let buffer: image::ImageBuffer<image::Rgb<u8>, Vec<u8>> = image::ImageBuffer::from_raw(w as u32,h as u32,pixels).ok_or_else(|| anyhow::anyhow!("raw to image"))?;
    buffer.save(p)?; Ok(())
}

let device = Device::Cpu;
let height = 64usize; let width = 96usize;
let mut data = Vec::with_capacity(3*height*width);
for y in 0..height { for x in 0..width { data.push(((x as f32/width as f32)*255.0) as f32); } }
for y in 0..height { for _x in 0..width { data.push(((y as f32/height as f32)*255.0) as f32); } }
for y in 0..height { for x in 0..width { data.push((((x+y) as f32/ (width+height) as f32)*255.0) as f32); } }
let r = Tensor::from_vec(data[0..height*width].to_vec(), (height,width), &device)?;
let g = Tensor::from_vec(data[height*width..2*height*width].to_vec(), (height,width), &device)?;
let b = Tensor::from_vec(data[2*height*width..].to_vec(), (height,width), &device)?;
let rgb = Tensor::stack(&[&r,&g,&b],0)?.to_dtype(DType::U8)?;
std::fs::create_dir_all("images_store")?;
save_image(&rgb, "images_store/test_gradient.png")?;
println!("Saved images_store/test_gradient.png");

Saved images_store/test_gradient.png


In [5]:
// Additional gradient test (pure candle)
use candle_core::{Tensor, DType};
let h = 140usize; let w = 200usize; let device = candle_core::Device::Cpu;
let mut r = Vec::with_capacity(h*w); let mut g = Vec::with_capacity(h*w); let mut b = Vec::with_capacity(h*w);
for y in 0..h { for x in 0..w { r.push((x as f32 / w as f32 * 255.0) as f32); g.push((y as f32 / h as f32 * 255.0) as f32); b.push((((x+y) as f32)/(w+h) as f32 * 255.0) as f32); } }
let rt = Tensor::from_vec(r, (h,w), &device)?;
let gt = Tensor::from_vec(g, (h,w), &device)?;
let bt = Tensor::from_vec(b, (h,w), &device)?;
let rgb = Tensor::stack(&[&rt,&gt,&bt],0)?.to_dtype(DType::U8)?;
std::fs::create_dir_all("images_store")?;
// Reuse save logic from first cell via writing a tiny inline helper again (cells isolated)
fn save_image<P: AsRef<std::path::Path>>(img: &Tensor, p: P) -> anyhow::Result<()> {
    let (c,h,w) = img.dims3()?; if c!=3 { anyhow::bail!("expected (3,H,W)"); }
    let flat = img.permute((1,2,0))?.flatten_all()?;
    let pixels = flat.to_vec1::<u8>()?;
    let buf: image::ImageBuffer<image::Rgb<u8>, Vec<u8>> = image::ImageBuffer::from_raw(w as u32,h as u32,pixels).ok_or_else(|| anyhow::anyhow!("raw to image"))?;
    buf.save(p)?; Ok(())
}
save_image(&rgb, "images_store/test_gradient2.png")?;
println!("Saved images_store/test_gradient2.png");

Saved images_store/test_gradient2.png
